# TMC-PINN Experiments
Run this notebook top to bottom. Each cell is a step.

**Order:**
1. Setup
2. Smoke test (100 epochs — confirms everything works)
3. Full run — all 7 conditions on reaction
4. Generate paper figures
5. Back up results

## Cell 1 — Install dependencies & check GPU

In [ ]:
import subprocess, sys

# Install required packages
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'torch', 'numpy', 'matplotlib', 'tqdm', 'pandas', 'scipy',
                '--break-system-packages', '-q'], check=False)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected')

## Cell 2 — Clone repo (skip if already cloned)

In [ ]:
import os

if not os.path.exists('PInnns'):
    os.system('git clone https://github.com/michae6345-crypto/PInnns.git')
    print('Cloned.')
else:
    os.system('cd PInnns && git pull origin main')
    print('Already exists — pulled latest.')

os.chdir('PInnns')
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

## Cell 3 — Import training script

In [ ]:
# Import main() from train_pinn.py so we can call it directly
import importlib.util, sys

spec = importlib.util.spec_from_file_location('train_pinn', './train_pinn.py')
tp   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tp)

print('train_pinn.py loaded successfully')

## Cell 4 — SMOKE TEST
**Run this before anything else.** 100 epochs, takes ~1 minute.
If this completes without errors, you are good to run the full experiments.

In [ ]:
print('Running smoke test: fp64 lbfgs, 100 epochs...')

tp.main(
    pde          = 'reaction',
    dtype_start  = 'fp64',
    optim_start  = 'lbfgs',
    total_epochs = 100,
    out_dir      = './results',
)

import os
files = os.listdir('./results/reaction/fp64lbfgs/')
print('\nSmoke test PASSED. Output files:')
for f in sorted(files):
    print(f'  {f}')

## Cell 5 — FULL RUN: all 7 conditions on reaction
**Only run this after the smoke test passes.**
Expected time on A10: 2-4 hours total. Do not close the browser tab.

In [ ]:
import time

# ============================================================
# SETTINGS — only change these if needed
TOTAL_EPOCHS = 2000    # same for all conditions (fixed budget)
SWITCH_EPOCH = 1000    # halfway point for switching conditions
PDE          = 'reaction'
DEVICE       = 'cuda:0'
OUT_DIR      = './results'
# ============================================================

# All 7 conditions: (dtype_start, optim_start, dtype_switch, optim_switch)
# For static baselines, dtype_switch and optim_switch are None
CONDITIONS = [
    # --- 4 static baselines ---
    ('fp64', 'lbfgs', None,   None),    # best expected
    ('fp64', 'adam',  None,   None),
    ('fp32', 'lbfgs', None,   None),
    ('fp32', 'adam',  None,   None),    # worst expected
    # --- 3 switching conditions ---
    ('fp32', 'adam',  'fp64', 'adam'),  # same optimizer, precision up
    ('fp32', 'lbfgs', 'fp64', 'lbfgs'),# same optimizer, precision up
    ('fp32', 'adam',  'fp64', 'lbfgs'),# cross-type — KEY condition
]

total_start = time.time()

for i, (ds, os_, dw, ow) in enumerate(CONDITIONS):
    label = f'{ds}{os_}' if dw is None else f'{ds}{os_}_to_{dw}{ow}'
    print(f'\n[{i+1}/7] Running: {label}')
    t0 = time.time()

    kwargs = dict(
        pde          = PDE,
        device       = DEVICE,
        dtype_start  = ds,
        optim_start  = os_,
        total_epochs = TOTAL_EPOCHS,
        out_dir      = OUT_DIR,
    )
    if dw is not None:
        kwargs['dtype_switch']  = dw
        kwargs['optim_switch']  = ow
        kwargs['switch_epoch']  = SWITCH_EPOCH

    tp.main(**kwargs)

    elapsed = time.time() - t0
    print(f'  Done in {elapsed/60:.1f} min')

total = time.time() - total_start
print(f'\nAll 7 conditions complete in {total/60:.1f} min total')

## Cell 6 — Check all 7 result folders exist

In [ ]:
import os

expected = [
    'fp64lbfgs', 'fp64adam', 'fp32lbfgs', 'fp32adam',
    'fp32adam_to_fp64adam', 'fp32lbfgs_to_fp64lbfgs', 'fp32adam_to_fp64lbfgs'
]

print('Results check:')
all_good = True
for cond in expected:
    path = f'./results/reaction/{cond}'
    if os.path.exists(path):
        files = os.listdir(path)
        csvs  = [f for f in files if f.endswith('.csv')]
        print(f'  OK  {cond} — {len(csvs)} CSVs')
    else:
        print(f'  MISSING  {cond}')
        all_good = False

print('\nAll results present!' if all_good else '\nSome conditions missing — check errors above.')

## Cell 7 — Generate paper figures and LaTeX table

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('aggregate_results', './aggregate_results.py')
ag   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ag)

import types
args = types.SimpleNamespace(results_dir='./results', out_dir='./paper_figures')

import os; os.makedirs('./paper_figures', exist_ok=True)

import pandas as pd
eval_df = ag.load_eval_logs('./results')
loss_df = ag.load_loss_logs('./results')

ag.build_summary_table(eval_df, './paper_figures')
ag.plot_loss_curves(loss_df, eval_df, './paper_figures')
ag.plot_l2_bars(eval_df, './paper_figures')
ag.plot_timing_bars(eval_df, './paper_figures')

print('\nPaper figures saved to ./paper_figures/')
print('Files:', os.listdir('./paper_figures/'))

## Cell 8 — Back up results
**Run this before closing Lambda.** Downloads a zip of all results.

In [ ]:
import os, zipfile, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_name  = f'results_backup_{timestamp}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['./results', './paper_figures']:
        for root, dirs, files in os.walk(folder):
            for file in files:
                filepath = os.path.join(root, file)
                zf.write(filepath)

size_mb = os.path.getsize(zip_name) / 1e6
print(f'Backup created: {zip_name}  ({size_mb:.1f} MB)')
print('Download this file from the Jupyter file browser (left sidebar) and upload to Drive.')